In [5]:
import re
from lxml import etree
import html
import os

# --- Configuration ---
# Directory containing your XML files
XML_DIR = '../GRC_misc/'
# Directory where you want to save the HTML output
OUTPUT_DIR = './html_output_hermann/'
# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# List of play identifiers (used to build filenames)
# Add other play IDs if you have corresponding text and app crit files
PLAY_IDS = ['ag','lib','eum','supp','seven','pb','pers'] # Currently only processing Agamemnon based on provided files

# XML namespaces (kept for reference, but XPath will ignore them)
ns = {'tei': 'http://www.tei-c.org/ns/1.0'}

print(f"Processing plays: {', '.join(PLAY_IDS)}")
print(f"HTML output will be saved in '{OUTPUT_DIR}'.")

Processing plays: ag, lib, eum, supp, seven, pb, pers
HTML output will be saved in './html_output_hermann/'.


In [6]:
# --- Helper Function (Modified for local-name() and lemma styling) ---
# Now simplified as highlighting is done *before* this function is called for Greek text
def get_node_html_content(node, mark_lemma_not_found=False):
    """Recursively get text content, handling specific tags and lemma styling for app crit."""
    parts = []
    # Prepend node's own text if it exists
    if node.text:
        parts.append(html.escape(node.text))
        
    for child in node:
        child_local_name = etree.QName(child.tag).localname if isinstance(child.tag, str) else None
        
        # Specific tag handling
        if child_local_name == 'note' and child.get('type') == 'editorial':
            parts.append(f'<span class="editorial-note">{html.escape("".join(child.itertext()))}</span>')
        elif child_local_name == 'gap':
            parts.append('<span class="gap">[...]</span>') 
        elif child_local_name == 'stage':
             parts.append(f'<span class="stage-direction">({html.escape("".join(child.itertext()))})</span>')
        elif child_local_name == 'lb': 
             parts.append('<br>')
        elif child_local_name == 'lem': 
            lemma_class = "app-lemma"
            if mark_lemma_not_found:
                lemma_class += " lemma-not-found" 
            parts.append(f'<span class="{lemma_class}">{html.escape("".join(child.itertext()))}</span>')
        elif child_local_name == 'quote': 
             parts.append(f'<span class="app-quote">"{html.escape("".join(child.itertext()))}"</span>')
        # Recursive call for other element nodes
        elif isinstance(child.tag, str): 
             parts.append(get_node_html_content(child, mark_lemma_not_found)) 
        # Ignore comments, PIs
        else:
             pass

        # Append tail text of the child
        if child.tail:
            parts.append(html.escape(child.tail))
            
    return "".join(parts)

# --- New Helper Function for Raw Text Extraction ---
def get_raw_text(node):
    """Recursively get raw text content, ignoring specific tags like notes, stage directions."""
    parts = []
    if node.text:
        parts.append(node.text)
    for child in node:
        child_local_name = etree.QName(child.tag).localname if isinstance(child.tag, str) else None
        # Exclude certain tags from raw text extraction
        if child_local_name not in ['note', 'stage', 'gap', 'lb']:
             if isinstance(child.tag, str):
                 parts.append(get_raw_text(child))
        if child.tail:
            parts.append(child.tail)
    return "".join(parts).strip() # Ensure stripping happens here

In [7]:
# --- Core Processing Function (Added Lemma Check & Greek Highlighting) ---
def process_play(text_xml_filepath, app_crit_xml_filepath, html_output_filepath):
    """
    Parses Aeschylus Hermann TEI XML files (text and app crit),
    extracts relevant data, checks lemmas, highlights Greek text, 
    and generates a two-column HTML file.
    Uses local-name() in XPath to ignore namespaces.
    """
    print(f"\nProcessing '{os.path.basename(text_xml_filepath)}' and '{os.path.basename(app_crit_xml_filepath)}'...")
    try:
        # --- 1. Parsing ---
        parser = etree.XMLParser(remove_blank_text=True, recover=True)
        text_root = etree.parse(text_xml_filepath, parser)
        app_root = etree.parse(app_crit_xml_filepath, parser)
        print("   - XML files parsed.")

        # --- 2. Data Extraction ---
        greek_lines_raw_nodes = {} # Store original <l> nodes
        greek_text_raw_content = {} # Store RAW text content for checking
        app_crit_col = {}          # Store final HTML for app crit column
        lemmas_by_line = {}        # Store list of lemma texts for each Greek line number
        greek_col_final_html = {}  # Store final HTML for Greek column (with highlights)

        # --- Step A: Extract Raw Greek Text and Nodes ---
        greek_lines_found = text_root.xpath('.//*[local-name()="body"]//*[local-name()="l"][@n]')
        print(f"   - DEBUG: Found {len(greek_lines_found)} <l> elements with 'n' attribute using local-name().")
        for line in greek_lines_found:
            line_num_str = line.get('n')
            if line_num_str and line_num_str.isdigit():
                raw_text = get_raw_text(line) 
                greek_text_raw_content[line_num_str] = raw_text 
                greek_lines_raw_nodes[line_num_str] = line 

        # --- Step B: Extract App Crit, Collect Lemmas, Check Lemmas ---
        app_crit_notes_found = app_root.xpath('.//p[@n]') 
        print(f"   - DEBUG: Found {len(app_crit_notes_found)} <p> elements with 'n' attribute in app crit file.")
        lemmas_not_found_count = 0
        
        for p_note in app_crit_notes_found:
            line_num_str_attr = p_note.get('n')
            line_nums = []
            # (Line number parsing logic - same as before)
            if line_num_str_attr:
                if line_num_str_attr == 'DramatisPersonae' or 'note' in line_num_str_attr: continue
                if '-' in line_num_str_attr:
                    try:
                        start_end = line_num_str_attr.split('-')
                        if len(start_end) == 2 and start_end[0].isdigit() and start_end[1].isdigit():
                             start, end = map(int, start_end)
                             if start <= end: line_nums = [str(i) for i in range(start, end + 1)]
                             else: continue
                        else: continue
                    except ValueError:
                         if line_num_str_attr.isdigit(): line_nums = [line_num_str_attr]
                         else: continue
                elif line_num_str_attr.isdigit(): line_nums = [line_num_str_attr]
                else: continue 

            if line_nums:
                mark_lemma_red = False 
                lem_elements = p_note.xpath('.//*[local-name()="lem"]')
                if lem_elements:
                    lem_element = lem_elements[0] 
                    lem_text = "".join(lem_element.itertext()).strip()
                    if lem_text: 
                        lemma_found_in_any_line = False
                        for num_str in line_nums:
                             lemmas_by_line.setdefault(num_str, []).append(lem_text)
                             greek_line_text = greek_text_raw_content.get(num_str, "")
                             if lem_text in greek_line_text:
                                 lemma_found_in_any_line = True
                        if not lemma_found_in_any_line:
                            mark_lemma_red = True
                            lemmas_not_found_count += 1
                
                app_text_html = get_node_html_content(p_note, mark_lemma_not_found=mark_lemma_red).strip()
                if not app_text_html: continue

                for num_str in line_nums:
                     if num_str.isdigit():
                          html_content = f'<div class="line" id="app-{num_str}"><a class="line-num" data-line="{num_str}">{num_str}</a> <span class="text">{app_text_html}</span></div>'
                          app_crit_col.setdefault(num_str, []).append(html_content)

        # --- Step C: Generate Final Greek HTML with Highlights ---
        for line_num_str, line_node in greek_lines_raw_nodes.items():
            line_content_html_base = get_node_html_content(line_node).strip()
            highlighted_line_content = line_content_html_base
            if line_num_str in lemmas_by_line:
                unique_lemmas = set(lemmas_by_line[line_num_str])
                sorted_lemmas = sorted(list(unique_lemmas), key=len, reverse=True)
                temp_highlighted_content = highlighted_line_content
                for lem_text in sorted_lemmas:
                    if lem_text:
                        escaped_lem = html.escape(lem_text)
                        pattern = re.compile(f'(?<![>"\'])(?<!class=")({re.escape(escaped_lem)})(?![<"\'])') 
                        temp_highlighted_content = pattern.sub(f'<span class="matched-lemma">\g<1></span>', temp_highlighted_content)
                highlighted_line_content = temp_highlighted_content

            speaker_html = ''
            parent_sp = line_node.xpath('ancestor::*[local-name()="sp"][1]')
            if parent_sp:
                parent_sp_node = parent_sp[0]
                first_l = parent_sp_node.xpath('.//*[local-name()="l"][@n][1]') 
                if first_l and first_l[0] is line_node:
                    speaker_tag = parent_sp_node.xpath('./*[local-name()="speaker"][1]')
                    if speaker_tag and speaker_tag[0].text:
                        speaker_html = f'<span class="speaker">{html.escape(speaker_tag[0].text.strip())}</span> '
            
            final_html = f'<div class="line" id="g-{line_num_str}"><a class="line-num" data-line="{line_num_str}">{line_num_str}</a> <span class="text">{speaker_html}{highlighted_line_content}</span></div>'
            greek_col_final_html[line_num_str] = final_html

        # --- Reporting --- 
        actual_greek_lines = len(greek_col_final_html)
        actual_app_crit_entries = sum(len(v) for v in app_crit_col.values())
        print(f"   - Data processed: {actual_greek_lines} Greek lines, {actual_app_crit_entries} app crit entries.")
        print(f"   - Lemmas marked red in app crit (not found in text): {lemmas_not_found_count}")
        if actual_greek_lines == 0: print("   - CRITICAL WARNING: No Greek lines were processed.")

        # --- 3. HTML Generation --- 
        max_line_num = 0
        all_keys = list(greek_col_final_html.keys()) + list(app_crit_col.keys())
        valid_keys = [int(k) for k in all_keys if isinstance(k, str) and k.isdigit()] + [k for k in all_keys if isinstance(k, int)]
        if valid_keys: max_line_num = max(valid_keys)
        else: max_line_num = 0 

        greek_html_lines = []
        start_line = 1 
        for i in range(start_line, max_line_num + 1):
            line_key = str(i)
            greek_html_lines.append(greek_col_final_html.get(line_key, f'<div class="line empty" id="g-{line_key}"><a class="line-num" data-line="{line_key}">{line_key}</a> <span class="text"></span></div>'))
        greek_html = "\n".join(greek_html_lines)

        app_crit_lines = []
        valid_app_keys = [k for k in app_crit_col.keys() if isinstance(k, str) and k.isdigit()]
        sorted_app_keys = sorted(valid_app_keys, key=int)
        for line_key in sorted_app_keys:
            app_crit_lines.extend(app_crit_col[line_key])
        app_crit_html = "\n".join(app_crit_lines)

        title_tag_list = text_root.xpath('.//*[local-name()="teiHeader"]/*[local-name()="titleStmt"]/*[local-name()="title"]')
        main_title = title_tag_list[0].text if title_tag_list and title_tag_list[0].text is not None else os.path.basename(text_xml_filepath)
        author = "Aeschylus"
        editor = "Gottfried Hermann"
        pub_date = "1852"

        # --- HTML Template (Script section simplified) ---
        html_template = f"""
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
            <title>{main_title} (Hermann Edition)</title>
            <style>
                body {{
                    font-family: 'Georgia', serif; display: flex; flex-direction: column;
                    height: 100vh; margin: 0; background-color: #fdfdfd;
                }}
                header {{
                    padding: 10px 20px; border-bottom: 2px solid #ddd;
                    background-color: #fff; text-align: center; flex-shrink: 0;
                }}
                h1 {{ margin: 0; font-size: 1.8em; color: #333; }}
                h2 {{ margin: 5px 0 0; font-size: 1.1em; color: #555; font-weight: normal;}}
                .container {{
                    display: grid; grid-template-columns: repeat(2, 1fr); 
                    gap: 15px; 
                    flex-grow: 1; padding: 15px; overflow: hidden; 
                }}
                .column {{
                    background-color: #ffffff; border: 1px solid #e0e0e0; border-radius: 4px;
                    display: flex; flex-direction: column; overflow: hidden; 
                }}
                .column h3 {{
                    text-align: center; margin: 0; padding: 12px;
                    border-bottom: 1px solid #e0e0e0; background-color: #f0f0f0; 
                    color: #444; font-size: 1em; font-weight: bold;
                    flex-shrink: 0;
                }}
                .content {{
                     padding: 10px 15px;
                     overflow-y: auto; 
                     height: 100%; 
                     line-height: 1.6; 
                     flex-grow: 1;
                     scroll-behavior: smooth;
                }}
                .line {{
                    display: flex; align-items: baseline; padding: 2px 5px; 
                    border-radius: 3px;
                     margin-bottom: 2px; 
                     min-height: 1.5em;
                }}
                #app-crit-content .line {{
                     min-height: auto;
                     align-items: flex-start; 
                }}
                #greek-content .line.empty .text {{
                     color: #ccc; 
                }}
                .line.highlight {{ background-color: #e7f5ff; }}
                .line-num {{
                    flex-shrink: 0; width: 40px; font-size: 0.75em; 
                    color: #999; 
                    cursor: pointer; text-align: right; margin-right: 12px; 
                    font-family: monospace;
                    padding-top: 0.2em; 
                }}
                .line-num:hover {{ color: #007bff; text-decoration: underline; }}
                .text {{ flex-grow: 1; }} 
                .speaker {{ font-weight: bold; margin-right: 8px; color: #800000; }}
                .app-lemma {{ font-style: italic; color: #0056b3; background-color: #f0f8ff; padding: 0 3px; border-radius: 2px;}}
                .app-quote {{ font-family: monospace; color: #444; }}
                .editorial-note {{ color: #006400; font-style: italic; }}
                .gap {{ color: #999; font-style: italic; }}
                 .stage-direction {{ color: #777; font-style: italic; font-size: 0.9em; }}
                 #app-crit-content .text {{
                      word-break: break-word;
                      white-space: normal; 
                 }}
                .lemma-not-found {{ 
                    color: red !important; 
                }}
                .matched-lemma {{
                    background-color: #fffacd; /* Light yellow */
                }}
            </style>
        </head>
        <body>
            <header>
                <h1>{main_title}</h1>
                <h2>{author} | Edited by: {editor} ({pub_date})</h2>
            </header>
            <div class="container">
                <div class="column">
                    <h3>Greek Text</h3>
                    <div class="content" id="greek-content">{greek_html}</div>
                </div>
                <div class="column">
                    <h3>Apparatus Criticus</h3>
                    <div class="content" id="app-crit-content">{app_crit_html}</div>
                </div>
            </div>
            <script>
                // --- SIMPLIFIED SCRIPT (Removed scroll listener) --- 
                document.addEventListener('DOMContentLoaded', function() {{
                    const container = document.querySelector('.container');
                    const contents = {{ // Store content divs by ID for easy access
                        'g': document.getElementById('greek-content'),
                        'app': document.getElementById('app-crit-content')
                    }};
                    let lastClickedLine = null;

                    // Function to scroll a specific column to an element
                    function scrollToElementInColumn(columnPrefix, lineNumber) {{
                        const elementId = `${{columnPrefix}}-${{lineNumber}}`; // Use template literal
                        const element = document.getElementById(elementId);
                        const contentDiv = contents[columnPrefix];

                        if (element && contentDiv) {{
                            element.classList.add('highlight');
                            const elementTop = element.offsetTop - contentDiv.offsetTop;
                            const elementHeight = element.offsetHeight;
                            const contentHeight = contentDiv.clientHeight;
                            let scrollToPosition = elementTop - (contentHeight / 2) + (elementHeight / 2);
                            scrollToPosition = Math.max(0, scrollToPosition);
                            scrollToPosition = Math.min(contentDiv.scrollHeight - contentHeight, scrollToPosition);
                            
                            // Directly set scrollTop for immediate jump, or use behavior:'smooth' if supported and desired
                            contentDiv.scrollTop = scrollToPosition;
                            
                            // Alternative smooth scroll:
                            // contentDiv.scrollTo({{ top: scrollToPosition, behavior: 'smooth' }});
                        }}
                    }}

                    container.addEventListener('click', function(event) {{
                        if (event.target.classList.contains('line-num')) {{
                            event.preventDefault();
                            const lineNumber = event.target.dataset.line;

                            // Remove previous highlights
                            document.querySelectorAll(`.line.highlight`).forEach(el => el.classList.remove('highlight'));

                            // Scroll both columns to the target line number
                            scrollToElementInColumn('g', lineNumber);
                            scrollToElementInColumn('app', lineNumber);

                            lastClickedLine = lineNumber; // Update last clicked line
                        }}
                    }});
                }});
            </script>
        </body>
        </html>
        """

        # --- 4. Write to file ---
        with open(html_output_filepath, 'w', encoding='utf-8') as f:
            f.write(html_template)
        print(f"   - Successfully created '{os.path.basename(html_output_filepath)}'.")

    # --- Error Handling --- 
    except FileNotFoundError as e:
        print(f"   - Error: File not found. {e}")
    except etree.XMLSyntaxError as e:
        print(f"   - Error: XML syntax error processing '{e.filename}'. {e}")
    except Exception as e:
        print(f"   - Error: An unexpected error occurred. {e}")
        import traceback
        traceback.print_exc() 

In [8]:
# --- Main Execution Loop ---
for play_id in PLAY_IDS:
    text_file = f'aesch.{play_id}.hermann1852-2.xml' 
    app_crit_file = f'aesch.{play_id}.appcrit-hermann1852.xml'
    output_file = f'aeschylus_{play_id}_hermann.html'

    text_xml_path = os.path.join(XML_DIR, text_file)
    app_crit_xml_path = os.path.join(XML_DIR, app_crit_file)
    output_html_path = os.path.join(OUTPUT_DIR, output_file)

    if not os.path.exists(text_xml_path):
        print(f"\nSkipping '{play_id}': Text file not found at '{text_xml_path}'")
        continue
    if not os.path.exists(app_crit_xml_path):
        print(f"\nSkipping '{play_id}': App crit file not found at '{app_crit_xml_path}'")
        continue

    process_play(text_xml_path, app_crit_xml_path, output_html_path)

print("\nProcessing complete for available Hermann plays.")


Processing 'aesch.ag.hermann1852-2.xml' and 'aesch.ag.appcrit-hermann1852.xml'...
   - XML files parsed.
   - DEBUG: Found 1644 <l> elements with 'n' attribute using local-name().
   - DEBUG: Found 944 <p> elements with 'n' attribute in app crit file.
   - Data processed: 1644 Greek lines, 1092 app crit entries.
   - Lemmas marked red in app crit (not found in text): 524
   - Successfully created 'aeschylus_ag_hermann.html'.

Processing 'aesch.lib.hermann1852-2.xml' and 'aesch.lib.appcrit-hermann1852.xml'...
   - XML files parsed.
   - DEBUG: Found 1073 <l> elements with 'n' attribute using local-name().
   - DEBUG: Found 695 <p> elements with 'n' attribute in app crit file.
   - Data processed: 1073 Greek lines, 830 app crit entries.
   - Lemmas marked red in app crit (not found in text): 193
   - Successfully created 'aeschylus_lib_hermann.html'.

Processing 'aesch.eum.hermann1852-2.xml' and 'aesch.eum.appcrit-hermann1852.xml'...
   - XML files parsed.
   - DEBUG: Found 1039 <l> ele